## **Expiry-Day Liquidity Concentration in Indian Index Derivatives**

### **Notebook: 03: True Expiry Dates and Calendar Audit**

---

**Scope:** Takes expiry dates per index from the Bhavcopy expiry field and audits the old rule-based calendar.

---

### Pipeline sequence

1. **01_data_collection.ipynb** -- index OHLCV, F&O Bhavcopy archive (NOTE: the rule-based expiry calendar it builds is NOT used for inference)
2. **02_feature_engineering_and_fo_construction.ipynb** -- technical features (true-calendar expiry features) and the true-expiry index panel
3. **03_weekly_options_discontinuation_identification.ipynb** -- true expiry dates per index and a calendar audit
4. **04_primary_diff_in_diff_analysis.ipynb** -- corrected symbol-month DiD (turnover, expiry-day turnover, expiry-day share)
5. **05_robustness_checks.ipynb** -- placebo, window sensitivity, substitution range, participant mix, futures market quality
6. **06_expiry_volume_event_study.ipynb** -- expiry-day concentration for six NSE/BSE indices
7. **07_walk_forward_predictability.ipynb** -- secondary: leakage-free predictability test on the true calendar (null)
8. **08_interrupted_time_series.ipynb** -- secondary: settlement-weekday ITS plus clean weekday-only contrasts (null)
9. **09_figures.ipynb** -- displays the corrected figures
10. **10_option_microstructure_and_placebo.ipynb** -- option-level outcomes, stock-F&O placebo, dose-response

**Current location:** **03_weekly_options_discontinuation_identification.ipynb** <- *you are here*


In [1]:
import sys
from pathlib import Path

project_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

import pandas as pd
import json
from src import config

pd.set_option('display.max_columns', 20)
print('Project root:', project_dir)

Project root: E:\Research-IPR\Research_BN_N50\banknifty-expiry-volatility-ml


Rebuild with `python -m scripts.build_index_expiry_panel` (~15 min). The old calendar in `data/raw/nse_expiry_calendar.csv` mis-flags many expiries (audit below) and must not be used.

In [2]:
ex = pd.read_parquet(config.INTERIM_DATA_DIR / 'index_expiry_dates.parquet')
bn = ex[(ex.Symbol == 'BANKNIFTY') & ex.IsMonthly].copy()
bn['Weekday'] = bn.ExpiryDate.dt.day_name()
bn.tail(24)

,Symbol,ExpiryDate,IsMonthly,Weekday
252,BANKNIFTY,2024-10-30,True,Wednesday
255,BANKNIFTY,2024-11-27,True,Wednesday
256,BANKNIFTY,2024-12-24,True,Tuesday
257,BANKNIFTY,2025-01-30,True,Thursday
258,BANKNIFTY,2025-02-27,True,Thursday
259,BANKNIFTY,2025-03-27,True,Thursday
260,BANKNIFTY,2025-04-24,True,Thursday
261,BANKNIFTY,2025-05-29,True,Thursday
262,BANKNIFTY,2025-06-26,True,Thursday
263,BANKNIFTY,2025-07-31,True,Thursday


In [3]:
json.load(open(config.RESULTS_DIR / 'corrected_robustness.json'))['calendar_audit']

{'BANKNIFTY': {'true_monthly_expiries': 81,
  'matched_by_old_calendar': 71,
  'mismatched': 10,
  'mismatched_since_2024-11-14': 8},
 'NIFTY': {'true_monthly_expiries': 82,
  'matched_by_old_calendar': 64,
  'mismatched': 18,
  'mismatched_since_2024-11-14': 10}}

In [4]:
p = pd.read_parquet(config.INTERIM_DATA_DIR / 'index_expiry_panel.parquet')
p[p.Symbol == 'BANKNIFTY'].set_index('Date')['NExpiriesListed'].loc['2024-10-10':'2024-11-20']

Date
2024-10-10    9.0
2024-10-11    9.0
2024-10-14    9.0
2024-10-15    9.0
2024-10-16    9.0
2024-10-17    8.0
2024-10-18    8.0
2024-10-21    8.0
2024-10-22    8.0
2024-10-23    8.0
2024-10-24    7.0
2024-10-25    7.0
2024-10-28    7.0
2024-10-29    7.0
2024-10-30    7.0
2024-10-31    7.0
2024-11-01    7.0
2024-11-04    7.0
2024-11-05    7.0
2024-11-06    7.0
2024-11-07    6.0
2024-11-08    6.0
2024-11-11    6.0
2024-11-12    6.0
2024-11-13    6.0
2024-11-14    5.0
2024-11-18    5.0
2024-11-19    5.0
Name: NExpiriesListed, dtype: float64